# 2.1 Travel Agent — MCP remoto (adaptado a lab local)

**Convención de este fork:**
- 🔸 **ORIGINAL DEL CURSO**: celda original, comentada para que *Run All* no falle. Dice por qué no se corre.
- 🟢 **ADAPTADO LOCAL**: la versión que corre con Ollama, con el motivo del cambio.
- Celdas sin marca: iguales al curso.

**Diferencia con `2.1_mcp.ipynb`:** aquí el servidor MCP es **remoto de verdad** (Kiwi.com, `streamable_http` + `url`). No hay subproceso local ni `command`.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (está en la raíz del repo).
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [1]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-2/ y local_model.py en la raíz del repo.
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model

In [2]:
# Sin cambios: servidor MCP REMOTO (no hay subproceso; el cliente habla HTTP con Kiwi.com).
# No necesita API key. Requiere internet desde la máquina de Jupyter.
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [3]:
# 🟢 ADAPTADO LOCAL (celda nueva): ver qué tools expone el servidor remoto y sus parámetros.
# Útil para entender qué tiene que adivinar el modelo antes de llamarlas.
for t in tools:
    print(f"- {t.name}: {(t.description or '').splitlines()[0][:120]}")
    print("    args:", list(t.args.keys()))

- search-flight: # Search for flights
    args: ['flyFrom', 'flyTo', 'departureDate', 'departureDateFlexDays', 'departureDateTo', 'returnDate', 'returnDateFlexDays', 'returnDateTo', 'adults', 'children', 'infants', 'cabinClass', 'currency', 'locale', 'nights_in_dst_from', 'nights_in_dst_to', 'one_for_city', 'max_sector_stopovers', 'price_from', 'price_to', 'max_fly_duration', 'select_airlines', 'exclude_airlines', 'dtime_from', 'dtime_to', 'atime_from', 'atime_to', 'ret_dtime_from', 'ret_dtime_to', 'ret_atime_from', 'ret_atime_to', 'stopover_from', 'stopover_to', 'adults_hold_bags', 'adults_hand_bags', 'children_hold_bags', 'children_hand_bags', 'allow_self_transfer', 'allow_overnight_stopovers', 'allow_diff_airport_connection', 'stopover_airports', 'exclude_stopover_airports', 'stopover_countries', 'exclude_stopover_countries', 'fly_days', 'ret_fly_days', 'sort']
- feedback-to-devs: Send feedback, bug reports, or feature requests about the Kiwi.com search-flight MCP tool to its develo

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: gpt-5-nano es API de pago; se usa gemma4 en Ollama (soporta tools).
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
# from langgraph.checkpoint.memory import InMemorySaver
#
# agent = create_agent(
#     "gpt-5-nano",
#     tools=tools,
#     checkpointer=InMemorySaver(),
#     system_prompt="You are a travel agent. No follow up questions."
# )

In [4]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → get_model("gemma4:latest"): sin APIs de pago; gemma4 soporta tools y es el más rápido del lab.
# El checkpointer se mantiene: en notebook sí se usa InMemorySaver (en langgraph dev NO).
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=get_model("gemma4:latest"),
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: se reemplaza por stream_mode='updates' para ver cada paso (model → tools → model) mientras corre en CPU; el invoke mudo puede tardar minutos sin dar señales.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.messages import HumanMessage
#
# config = {"configurable": {"thread_id": "1"}}
#
# response = await agent.ainvoke(
#     {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
#     config
#     )

In [6]:
# 🟢 ADAPTADO LOCAL
# Motivo: en CPU cada paso tarda; con astream(stream_mode="updates") se ve qué tool llamó el modelo
# y con qué argumentos (fechas, aeropuertos) en vez de esperar un ainvoke mudo.
# Nota: es async (MCP) → astream / ainvoke, nunca invoke.
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}
question = HumanMessage(content="Get me a direct flight from Barranquilla to Tokyo on Octover 31st 2026")

async for step in agent.astream({"messages": [question]}, config, stream_mode="updates"):
    for node, upd in step.items():
        m = upd["messages"][-1]
        if getattr(m, "tool_calls", None):
            print(f"[{node}] tool_calls →", m.tool_calls)
        else:
            print(f"[{node}]", str(m.content)[:500])

# Estado final del thread (equivale a 'response' del original)
response = (await agent.aget_state(config)).values

[model] tool_calls → [{'name': 'search-flight', 'args': {'departureDate': '31/10/2026', 'flyFrom': 'Barranquilla', 'flyTo': 'Tokyo'}, 'id': 'aa5fe871-fd24-4064-ba13-c573df2720d5', 'type': 'tool_call'}]
[tools] [{'type': 'text', 'text': '{\n  "query": "Barranquilla → Tokyo on 31/10/2026, 1 adult",\n  "currency": "EUR",\n  "passengers": {\n    "adults": 1,\n    "children": 0,\n    "infants": 0\n  },\n  "resultsCount": 15,\n  "itineraries": [\n    {\n      "id": "031804cd511500001a026a93_0|04cd230e51160000ae9a03ef_0|230e209f51170000742b1528_0",\n      "price": 628.0,\n      "priceFormatted": "628 EUR",\n      "totalDurationSeconds": 208980,\n      "bookingUrl": "https://kiwi.com/u/s77wug7",\n      "image
[model] What is your departure city, destination city, and what date would you like to travel?


In [7]:
# Sin cambios
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st', additional_kwargs={}, response_metadata={}, id='a219d8f3-59ed-4a43-913e-ece7709bf4b9'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-04T23:40:50.8780021Z', 'done': True, 'done_reason': 'stop', 'total_duration': 45144984300, 'load_duration': 5854900, 'prompt_eval_count': 2292, 'prompt_eval_duration': 42693429000, 'eval_count': 40, 'eval_duration': 2413866000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10949-f9f4-7c63-a07a-f70c5891b467-0', tool_calls=[{'name': 'search-flight', 'args': {'departureDate': '31/03/2025', 'flyFrom': 'San Francisco', 'flyTo': 'Tokyo'}, 'id': '63117c83-2169-42f4-9429-e3324c253c2c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2292, 'output_tokens': 40, 'total_tokens': 2332}),
              ToolMessag

In [8]:
# Sin cambios
print(response["messages"][-1].content)

What is your departure city, destination city, and what date would you like to travel?


In [10]:
# 🟢 ADAPTADO LOCAL (celda nueva, opcional): prueba con una ruta propia.
# Mismo thread → el agente recuerda la conversación anterior (checkpointer).
question = HumanMessage(content="Now find me the cheapest flight from Barranquilla (BAQ) to Bogotá (BOG) next Friday, today is Octover 4, sunday 2026")

async for step in agent.astream({"messages": [question]}, config, stream_mode="updates"):
    for node, upd in step.items():
        m = upd["messages"][-1]
        print(f"[{node}]", m.tool_calls if getattr(m, "tool_calls", None) else str(m.content)[:800])

[model] [{'name': 'search-flight', 'args': {'departureDate': '10/10/2026', 'flyFrom': 'BAQ', 'flyTo': 'BOG', 'sort': 'price'}, 'id': '3a00c900-168d-45b4-81a3-2ba1fd973355', 'type': 'tool_call'}]
[tools] [{'type': 'text', 'text': '{\n  "query": "BAQ → BOG on 10/10/2026, 1 adult",\n  "currency": "EUR",\n  "passengers": {\n    "adults": 1,\n    "children": 0,\n    "infants": 0\n  },\n  "resultsCount": 15,\n  "itineraries": [\n    {\n      "id": "0318049b51000000ff854389_0",\n      "price": 90.0,\n      "priceFormatted": "90 EUR",\n      "totalDurationSeconds": 5400,\n      "bookingUrl": "https://kiwi.com/u/fgj3ts4",\n      "imageId": "bogota_co",\n      "baggage": {\n        "personalItem": 1,\n        "cabinBag": 0,\n        "checkedBag": 0\n      },\n      "outbound": {\n        "from": "BAQ",\n        "to": "BOG",\n        "departureTime": "2026-10-10T07:20:00",\n        "arrivalTime": "2026-10-10T08:50:00",\n        "durationSeconds": 5400,\n        "stops": 0,\n        "route": [\n  

In [11]:
# 🟢 ADAPTADO LOCAL
# Motivo: Kiwi devuelve JSON grande → con num_ctx=8192 Ollama recorta el prompt y el modelo pierde la pregunta.
tools_vuelos = [t for t in tools if t.name == "search-flight"]   # sin feedback-to-devs

agent = create_agent(
    model=get_model("gemma4:latest", num_ctx=32768),   # gemma4 admite hasta 128K
    tools=tools_vuelos,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions. Answer with the 3 best options: price, times, stops and booking link."
)
config = {"configurable": {"thread_id": "2"}}   # thread limpio, sin el historial anterior

In [12]:
# 🟢 ADAPTADO LOCAL (celda nueva, opcional): prueba con una ruta propia.
# Mismo thread → el agente recuerda la conversación anterior (checkpointer).
question = HumanMessage(content="Now find me the cheapest flight from Barranquilla (BAQ) to Bogotá (BOG) next Friday, today is Octover 4, sunday 2026")

async for step in agent.astream({"messages": [question]}, config, stream_mode="updates"):
    for node, upd in step.items():
        m = upd["messages"][-1]
        print(f"[{node}]", m.tool_calls if getattr(m, "tool_calls", None) else str(m.content)[:800])

[model] [{'name': 'search-flight', 'args': {'departureDate': '10/10/2026', 'flyFrom': 'BAQ', 'flyTo': 'BOG', 'sort': 'price'}, 'id': '064a7e71-f263-4e56-8634-6ca0003a4c3f', 'type': 'tool_call'}]
[tools] [{'type': 'text', 'text': '{\n  "query": "BAQ → BOG on 10/10/2026, 1 adult",\n  "currency": "EUR",\n  "passengers": {\n    "adults": 1,\n    "children": 0,\n    "infants": 0\n  },\n  "resultsCount": 15,\n  "itineraries": [\n    {\n      "id": "0318049b51000000ff854389_0",\n      "price": 90.0,\n      "priceFormatted": "90 EUR",\n      "totalDurationSeconds": 5400,\n      "bookingUrl": "https://kiwi.com/u/vzc8bk",\n      "imageId": "bogota_co",\n      "baggage": {\n        "personalItem": 1,\n        "cabinBag": 0,\n        "checkedBag": 0\n      },\n      "outbound": {\n        "from": "BAQ",\n        "to": "BOG",\n        "departureTime": "2026-10-10T07:20:00",\n        "arrivalTime": "2026-10-10T08:50:00",\n        "durationSeconds": 5400,\n        "stops": 0,\n        "route": [\n   